# Sentiment Analysis

In this notebook we use an RNN network to classify text. The text we will use are IMDB movie reviews and the classification is either positive (1) or negative (0). We employ several different techniques from our other models
* The input lengths vary and there are no windows -- the input is the entire review. This will require the use of "padding" tokens to keep the data stored in a matrix
* The input words are encoded to integers using a tensorflow text vectorization layer.
* These vectors are embedded in a high dimensional vector space using an embedding layer
* The model data is stored as a tensorflow dataset, which allows shuffling, batchig and caching to optimize training time.

In [ ]:
import tensorflow_datasets as tfds
import tensorflow as tf

Load the data from tensorflow's own database

In [ ]:
raw_train, raw_validation, raw_test = tfds.load(
    name="imdb_reviews",
    split=['train[:80%]', 'train[80%:]', 'test'],
    as_supervised=True
)
print(len(raw_train), len(raw_validation), len(raw_test))

Clean up html, whitespace, etc

In [ ]:
def clean_text(review, label):
    review = tf.strings.regex_replace(review, "<[^>]+>", " ")  # all HTML tags
    review = tf.strings.regex_replace(review, "\\s+", " ")     # collapse whitespace
    return review, label

raw_train = raw_train.map(clean_text)
raw_validation = raw_validation.map(clean_text)
raw_test = raw_test.map(clean_text)

Create datasets that use shuffling, batching and prefetching. (Change the batchsize here if you want)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_set = raw_train.cache().shuffle(20000).batch(32).prefetch(AUTOTUNE)
valid_set = raw_validation.cache().batch(32).prefetch(AUTOTUNE)
test_set = raw_test.cache().batch(32).prefetch(AUTOTUNE)

Show some of the raw data. Notice the use of `take` to extract data from a tf.dataset

In [ ]:
for review, label in raw_train.take(10):
    print(review.numpy().decode("utf-8")) # decode converts a python byte array to correct text
    print(label.numpy())

Do some EDA on the dataset. Also note how we access and iterate through the raw data

In [ ]:
import matplotlib.pyplot as plt

# Assuming 'train_set' is a tf.data.Dataset with labels (0 or 1)
label_list = []
for _, label in raw_train:
  label_list.append(label.numpy())

In [ ]:
%matplotlib inline


## make a histogram of the label_list

In [ ]:
review_lengths = []
for review, _ in raw_train:
  review_lengths.append(len(review.numpy()))

## make a histogram of review lengths

Is there a correlation between review length and positive/negative?

In [ ]:
import numpy as np

review_lengths = []
labels = []
for review, label in raw_train:
  review_lengths.append(len(review.numpy()))
  labels.append(label.numpy())

# Calculate correlation
correlation = np.corrcoef(review_lengths, labels)[0, 1]

# Plot review length vs label
plt.scatter(review_lengths, labels)
plt.xlabel("Review Length")
plt.ylabel("Label")
plt.title(f"Review Length vs Label (Correlation: {correlation:.2f})")
plt.show()

print(f"Correlation between review length and label: {correlation:.2f}")


As we have done before using scikit-learn (or manual code), we convert the top 10000 most common vocab terms in the dataset to integers. We will use a tensorflow TextVectorization layer to do so. The layer will take some time to train because of the size of the data. Therefore we show how to save the layer as a tensorflow model for extraction later. Set the 'false' below to 'true' to generate the layer. Be sure to update the filepath and mount Google Drive.

In [ ]:
import os
vocab_size = ## pick a vocab size

filepath = "TextVecLayer-IMBD.keras"
rebuild = True
if (rebuild or not os.path.exists(filepath)):
  text_vec_layer = tf.keras.layers.TextVectorization( ## define max_tokens here and set the output to "int"
  )
  text_vec_layer.adapt(raw_train.map(lambda review, label: review))
    
  # Create model to save TextVectorization Layer
  model = tf.keras.models.Sequential()
  model.add(tf.keras.Input(shape=(1,), dtype=tf.string))
  model.add(text_vec_layer)
    
  # Save.
  model.save(filepath)

Now load the layer (from above or an earlier session)

In [ ]:
loaded_model = tf.keras.models.load_model(filepath)
text_vec_layer = loaded_model.layers[0]

Let's see how this layer works

In [ ]:
text_vec_layer.vocabulary_size()

In [ ]:
",".join(text_vec_layer.get_vocabulary()[:100])

Token #1 is UNK, for unknown words (not in the top 1000 vocab). Token 0 is empty space used for padding.

In [ ]:
text_vec_layer(["you", 'saw', 'the', 'armadillo', '', '']).numpy()

Now create the model. Use the vector layer and an embedding layer. Then a simple GRU stack and a single neuron to classify. Note the use of sigmoid since the output data is a single [0,1] variable.

In [ ]:
import os, datetime

embed_size = 512
model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Embedding(
        input_dim=len(text_vec_layer.get_vocabulary()), 
        output_dim=embed_size,
        mask_zero=True
    ),

    ## add a GRU layer or 2
    ## optional dropout?
    ## last layer is Dense[1] with activation = sigmoid
])

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', 
    patience=3, 
    restore_best_weights=True
)

model.compile(loss="binary_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
history = model.fit(train_set, validation_data=valid_set, epochs=### set number of epochs, callbacks=[early_stopping])

### Task

*TO DO*. Run the model on some testing data and print a few of the samples it misclassifies. Read the samples yourself -- do you see why they were misclassified?

In [ ]:
for review, label in raw_test.take(50):
    truth = label.numpy()
    r = [review.numpy()]
    pred = model(tf.constant(r))[0][0].numpy()
    rprep = np.round(pred)
    if (int(truth) != int(rprep)):
        print(label.numpy(), pred)
        print(review.numpy().decode("utf-8")) # decode converts a python byte array to correct text
        print()

In [ ]:
model.evaluate(test_set)

In [ ]:
## Make a confusion matrix for the test set